Here we tackled the problem of building a predictive model for a response variable in a high-dimensional dataset with 999 predictors (plus one response), where 25% (≈250) of the predictors are categorical and some numerical variables exhibit low variance (e.g., limited levels like 1, 2, 3), posing challenges due to potential redundancy, noise, and computational complexity. In the *PCA + Lasso* workflow, we addressed this by preprocessing the data—encoding categorical variables with one-hot encoding (expanding to ~1,999 columns), filtering low-variance columns (reducing to ~1,700), standardizing the features, applying PCA to compress into 100-200 principal components capturing 95% variance, and using Lasso to select 30-50 predictive components, culminating in a linear regression model. In contrast, the *Tree-Based* workflow took a simpler route, label-encoding categoricals to maintain the original 999-column structure, applying a low-variance filter (reducing to ~950), skipping standardization (unnecessary for trees), training a Random Forest to rank feature importance, and selecting the top 20-50 original features for a final tree model. The key differences lie in the *PCA + Lasso* approach’s use of synthetic components (less interpretable, dimensionally reduced) versus the *Tree-Based* approach’s retention of original features (more interpretable, leveraging tree robustness to cardinality and scale), tailoring solutions to balance complexity, interpretability, and predictive power in a high-dimensional setting.

**Workflow 1: PCA + Lasso Approach**

**Assumptions**
- $n \times 999$ predictors, 1 response ($y$).
- 749 numerical, 250 categorical (25%).
- Some numericals have low variance (e.g., 1, 2, 3); categoricals may have low (e.g., yes/no) or high cardinality (e.g., 50 states).

**Step-by-Step**

**Handle Categorical Variables**  
*Description:* Convert 250 categorical columns to numerical format.  
- *Low Cardinality* (e.g., "yes/no", 2 levels):  
  - One-hot encode: "yes/no" $\to$ 2 columns (is_yes, is_no). Drop one (is_no) to avoid multicollinearity (dummy variable trap).  
  - *Expectation:* 1 column per variable (e.g., 250 $\to$ 250 binary columns).  
- *High Cardinality* (e.g., "state", 50 levels):  
  - One-hot encode: 50 states $\to$ 50 columns, but rare states (e.g., <1% frequency) may get filtered later.  
  - *Alternative:* Target encoding (replace "state" with mean ($y$) per state) if predicting ($y$) is priority—keeps it 1 column but risks leakage (use with cross-validation).  
  - *Expectation:* 250 $\to$ ~1,000-2,000 columns (avg 4-8 levels per variable).  
- *Tool:* `pandas.get_dummies()` or `sklearn.preprocessing.OneHotEncoder`.  
- *New Shape:* $n \times 1,999$ (749 numerical + 1,250 encoded categorical, assuming avg 5 levels).

**Variance Threshold**  
*Description:* Drop columns with near-zero variance (numerical or encoded).  
- *Low Variance Numericals* (e.g., 1, 2, 3):  
  - Variance $\approx$ 0.5-1. Set threshold low (e.g., 0.01) to keep them—they’re meaningful despite low spread.  
  - *Expectation:* Most survive unless nearly constant (e.g., 99% are 1).  
- *Low Cardinality Categoricals* (post-encoding): Binary columns (0/1) have variance = $p(1-p)$ (e.g., 10% 1’s $\to$ 0.09). Keep if >0.01.  
- *High Cardinality Categoricals* (post-encoding): Rare categories (e.g., 1% 1’s) have low variance (0.0099)—dropped if below threshold.  
- *Tool:* `sklearn.feature_selection.VarianceThreshold(threshold=0.01)`.  
- *New Shape:* $n \times 1,700$ (e.g., 300 columns dropped—rare categories or near-constant numericals).

**Standardize**  
*Description:* Scale all columns (numericals + encoded categoricals) to mean 0, std 1.  
- *Why:* PCA and Lasso need comparable scales.  
- *Low Variance Numericals:* [1, 2, 3] $\to$ [-1, 0, 1] (still useful).  
- *Categorical (encoded):* 0/1 columns $\to$ small negatives/positives (e.g., [-0.2, 0.8]).  
- *Tool:* `sklearn.preprocessing.StandardScaler`.  
- *Expectation:* Shape unchanged ($n \times 1,700$), values normalized.

**PCA**  
*Description:* Reduce dimensions by projecting onto principal components (PCs) capturing 95% variance.  
- *What Happens:* Combines 1,700 columns into ($k$) uncorrelated components.  
- *Low Variance Numericals:* Contribute to PCs if correlated with ($y$) or other features.  
- *Categorical (encoded):* High-cardinality variables (many 0/1 columns) often dominate variance; PCs blend them with numericals.  
- *Tool:* `sklearn.decomposition.PCA(n_components=0.95)`.  
- *Expectation:*  
  - *Shape:* $n \times k$ (e.g., $k = 100-200$, depending on correlations).  
  - *PCs:* Synthetic (e.g., PC1 = 0.3income + 0.2is_red - …).  
  - *Variance:* `pca.explained_variance_ratio_` sums to 0.95.

**Lasso Feature Selection**  
*Description:* Fit Lasso on PCA output to select predictive components.  
- *How:* $y = \beta_1 \cdot \text{PC1} + \beta_2 \cdot \text{PC2} + \cdots$, $\beta_j$ shrunk to 0 for irrelevant PCs.  
- *Tool:* `sklearn.linear_model.LassoCV(cv=5)` (auto-tunes $\lambda$).  
- *Expectation:*  
  - *Shape:* $n \times m$ (e.g., $m = 30-50$ PCs with non-zero $\beta$).  
  - *Result:* Best PCs for ($y$), blending original numerical/categorical info.

**Model & Evaluate**  
*Description:* Train a model (e.g., linear regression) on selected PCs, test performance.  
- *Tool:* `sklearn.linear_model.LinearRegression`, `cross_val_score`.  
- *Expectation:* $R^2$ or F1 (depending on ($y$))—e.g., 0.7+ if data’s predictive.**Workflow 1: PCA + Lasso Approach**

**Assumptions**
- $n \times 999$ predictors, 1 response ($y$).
- 749 numerical, 250 categorical (25%).
- Some numericals have low variance (e.g., 1, 2, 3); categoricals may have low (e.g., yes/no) or high cardinality (e.g., 50 states).

**Step-by-Step**

**Handle Categorical Variables**  
*Description:* Convert 250 categorical columns to numerical format.  
- *Low Cardinality* (e.g., "yes/no", 2 levels):  
  - One-hot encode: "yes/no" $\to$ 2 columns (is_yes, is_no). Drop one (is_no) to avoid multicollinearity (dummy variable trap).  
  - *Expectation:* 1 column per variable (e.g., 250 $\to$ 250 binary columns).  
- *High Cardinality* (e.g., "state", 50 levels):  
  - One-hot encode: 50 states $\to$ 50 columns, but rare states (e.g., <1% frequency) may get filtered later.  
  - *Alternative:* Target encoding (replace "state" with mean ($y$) per state) if predicting ($y$) is priority—keeps it 1 column but risks leakage (use with cross-validation).  
  - *Expectation:* 250 $\to$ ~1,000-2,000 columns (avg 4-8 levels per variable).  
- *Tool:* `pandas.get_dummies()` or `sklearn.preprocessing.OneHotEncoder`.  
- *New Shape:* $n \times 1,999$ (749 numerical + 1,250 encoded categorical, assuming avg 5 levels).

**Variance Threshold**  
*Description:* Drop columns with near-zero variance (numerical or encoded).  
- *Low Variance Numericals* (e.g., 1, 2, 3):  
  - Variance $\approx$ 0.5-1. Set threshold low (e.g., 0.01) to keep them—they’re meaningful despite low spread.  
  - *Expectation:* Most survive unless nearly constant (e.g., 99% are 1).  
- *Low Cardinality Categoricals* (post-encoding): Binary columns (0/1) have variance = $p(1-p)$ (e.g., 10% 1’s $\to$ 0.09). Keep if >0.01.  
- *High Cardinality Categoricals* (post-encoding): Rare categories (e.g., 1% 1’s) have low variance (0.0099)—dropped if below threshold.  
- *Tool:* `sklearn.feature_selection.VarianceThreshold(threshold=0.01)`.  
- *New Shape:* $n \times 1,700$ (e.g., 300 columns dropped—rare categories or near-constant numericals).

**Standardize**  
*Description:* Scale all columns (numericals + encoded categoricals) to mean 0, std 1.  
- *Why:* PCA and Lasso need comparable scales.  
- *Low Variance Numericals:* [1, 2, 3] $\to$ [-1, 0, 1] (still useful).  
- *Categorical (encoded):* 0/1 columns $\to$ small negatives/positives (e.g., [-0.2, 0.8]).  
- *Tool:* `sklearn.preprocessing.StandardScaler`.  
- *Expectation:* Shape unchanged ($n \times 1,700$), values normalized.

**PCA**  
*Description:* Reduce dimensions by projecting onto principal components (PCs) capturing 95% variance.  
- *What Happens:* Combines 1,700 columns into ($k$) uncorrelated components.  
- *Low Variance Numericals:* Contribute to PCs if correlated with ($y$) or other features.  
- *Categorical (encoded):* High-cardinality variables (many 0/1 columns) often dominate variance; PCs blend them with numericals.  
- *Tool:* `sklearn.decomposition.PCA(n_components=0.95)`.  
- *Expectation:*  
  - *Shape:* $n \times k$ (e.g., $k = 100-200$, depending on correlations).  
  - *PCs:* Synthetic (e.g., PC1 = 0.3income + 0.2is_red - …).  
  - *Variance:* `pca.explained_variance_ratio_` sums to 0.95.

**Lasso Feature Selection**  
*Description:* Fit Lasso on PCA output to select predictive components.  
- *How:* $y = \beta_1 \cdot \text{PC1} + \beta_2 \cdot \text{PC2} + \cdots$, $\beta_j$ shrunk to 0 for irrelevant PCs.  
- *Tool:* `sklearn.linear_model.LassoCV(cv=5)` (auto-tunes $\lambda$).  
- *Expectation:*  
  - *Shape:* $n \times m$ (e.g., $m = 30-50$ PCs with non-zero $\beta$).  
  - *Result:* Best PCs for ($y$), blending original numerical/categorical info.

**Model & Evaluate**  
*Description:* Train a model (e.g., linear regression) on selected PCs, test performance.  
- *Tool:* `sklearn.linear_model.LinearRegression`, `cross_val_score`.  
- *Expectation:* $R^2$ or F1 (depending on ($y$))—e.g., 0.7+ if data’s predictive.

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression

# Simulate a dataset: 1000 rows, 10 predictors (7 numerical, 3 categorical), 1 response
np.random.seed(42)
n_samples = 1000
data = {
    'num1': np.random.normal(0, 1, n_samples),  # High variance numerical
    'num2': np.random.normal(10, 5, n_samples),  # High variance
    'num3': np.random.randint(1, 4, n_samples),  # Low variance (1, 2, 3)
    'num4': np.ones(n_samples) + np.random.normal(0, 0.01, n_samples),  # Near-constant
    'num5': np.random.uniform(0, 100, n_samples),  # High variance
    'num6': np.random.randint(1, 4, n_samples),  # Low variance (1, 2, 3)
    'num7': np.random.normal(50, 2, n_samples),  # Moderate variance
    'cat1': np.random.choice(['yes', 'no'], n_samples),  # Low cardinality (2 levels)
    'cat2': np.random.choice(['red', 'blue', 'green'], n_samples),  # Medium (3 levels)
    'cat3': np.random.choice([f'state_{i}' for i in range(10)], n_samples)  # High cardinality (10 levels)
}
X = pd.DataFrame(data)
y = X['num1'] + 2 * X['num3'] + np.where(X['cat1'] == 'yes', 1, 0) + np.random.normal(0, 1, n_samples)  # Response

# Step 1: Handle Categorical Variables
# Separate numerical and categorical columns
num_cols = ['num1', 'num2', 'num3', 'num4', 'num5', 'num6', 'num7']
cat_cols = ['cat1', 'cat2', 'cat3']
X_num = X[num_cols]
X_cat = X[cat_cols]

# One-hot encode categoricals (drop one level to avoid multicollinearity)
encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' for dummy trap
X_cat_encoded = pd.DataFrame(encoder.fit_transform(X_cat), columns=encoder.get_feature_names_out(cat_cols))
print(f"After encoding: {X_cat_encoded.shape[1]} categorical columns (from 3)")  # e.g., 12 columns (1+2+9)

# Combine numerical and encoded categorical
X_processed = pd.concat([X_num, X_cat_encoded], axis=1)
print(f"Shape after encoding: {X_processed.shape}")  # e.g., 1000 x 19 (7 num + 12 cat)

# Step 2: Variance Threshold
selector = VarianceThreshold(threshold=0.01)  # Low threshold to keep 1,2,3-like variables
X_filtered = selector.fit_transform(X_processed)
kept_features = X_processed.columns[selector.get_support()].tolist()
print(f"Shape after variance filter: {X_filtered.shape}")  # e.g., drops num4 (near-constant)
print(f"Kept features: {len(kept_features)}")

# Convert back to DataFrame for clarity
X_filtered_df = pd.DataFrame(X_filtered, columns=kept_features)

# Step 3: Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=kept_features)
print(f"Shape after scaling: {X_scaled.shape}")

# Step 4: PCA
pca = PCA(n_components=0.95)  # Keep 95% variance
X_pca = pca.fit_transform(X_scaled)
print(f"Shape after PCA: {X_pca.shape}")  # e.g., 1000 x 8 (fewer components)
print(f"Variance explained: {sum(pca.explained_variance_ratio_):.3f}")

# Step 5: Lasso Feature Selection
lasso = LassoCV(cv=5, random_state=42)  # Cross-validate to tune lambda
lasso.fit(X_pca, y)
selected_components = np.where(lasso.coef_ != 0)[0]
X_lasso_selected = X_pca[:, selected_components]
print(f"Shape after Lasso: {X_lasso_selected.shape}")  # e.g., 1000 x 5 (non-zero coefs)
print(f"Selected components: {len(selected_components)}")

# Step 6: Model & Evaluate
X_train, X_test, y_train, y_test = train_test_split(X_lasso_selected, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
test_score = model.score(X_test, y_test)
print(f"Cross-validated R²: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
print(f"Test R²: {test_score:.3f}")

After encoding: 12 categorical columns (from 3)
Shape after encoding: (1000, 19)
Shape after variance filter: (1000, 18)
Kept features: 18
Shape after scaling: (1000, 18)
Shape after PCA: (1000, 16)
Variance explained: 0.969
Shape after Lasso: (1000, 16)
Selected components: 16
Cross-validated R²: 0.790 (±0.028)
Test R²: 0.779


**Workflow 2: Tree-Based Approach (No PCA)**

**Assumptions**
- Same as above: 749 numerical, 250 categorical, low/high cardinality issues.

**Step-by-Step**

**Handle Categorical Variables**  
*Description:* Prepare categoricals for tree models (e.g., Random Forest, XGBoost).  
- *Low Cardinality* (e.g., "yes/no"):  
  - Label encode: "yes" = 0, "no" = 1 (trees don’t need one-hot for low levels).  
  - *Expectation:* 1 column per variable (250 stay 250).  
- *High Cardinality* (e.g., "state", 50 levels):  
  - Label encode: "CA" = 0, "NY" = 1, …, "WY" = 49 (trees handle high cardinality better than Lasso).  
  - *Alternative:* One-hot encode if <15 levels and software supports sparse data (e.g., XGBoost); else, avoid—explodes to 1,000+ columns.  
  - *Caution:* High cardinality can bias trees toward frequent splits—use target encoding sparingly (mean ($y$) per category, with cross-validation).  
- *Tool:* `sklearn.preprocessing.LabelEncoder` or `pandas.factorize`.  
- *New Shape:* $n \times 999$ (all columns numerical now).

**Variance Threshold**  
*Description:* Drop near-zero variance columns.  
- *Low Variance Numericals* (e.g., 1, 2, 3):  
  - Keep with low threshold (0.01)—trees can still split on them.  
- *Categorical (encoded):* If label-encoded, variance depends on category balance (e.g., 90% "yes" $\to$ low variance). Drop if <0.01.  
- *Tool:* `sklearn.feature_selection.VarianceThreshold(threshold=0.01)`.  
- *New Shape:* $n \times 950$ (e.g., 50 dropped—near-constant numericals or imbalanced categoricals).

**No Standardization**  
*Description:* Trees (e.g., Random Forest, XGBoost) don’t require scaling—splits are order-based, not magnitude-based.  
- *Low Variance Numericals:* [1, 2, 3] stay as-is—trees split at 1.5, 2.5, etc.  
- *Categorical (encoded):* [0, 1, 2] (e.g., "red, blue, green") used directly.  
- *Expectation:* Shape unchanged ($n \times 950$).

**Train Tree Model**  
*Description:* Fit a tree-based model to rank feature importance.  
- *How:* Random Forest or XGBoost computes importance (e.g., reduction in impurity per feature).  
- *Low Variance Numericals:* If predictive, they’ll rank high (e.g., splits on 1 vs. 2 matter).  
- *High Cardinality Categoricals:* May dominate splits—watch for overfitting (limit tree depth).  
- *Tool:* `sklearn.ensemble.RandomForestRegressor` or `xgboost.XGBRegressor`.  
- *Expectation:* Importance scores for 950 features (e.g., "income" = 0.15, "state" = 0.08).

**Feature Selection**  
*Description:* Pick top features based on importance.  
- *How:* Select top 20-50 (e.g., threshold at 1% importance or cumulative 90% contribution).  
- *Expectation:* Shape: $n \times 30-50$ (e.g., mix of numericals and encoded categoricals like "income," "state").  
- *Tool:* `feature_importances_` (Random Forest) or `feature_importances_` (XGBoost).

**Model & Evaluate**  
*Description:* Retrain on selected features, test performance.  
- *Tool:* Same tree model, `cross_val_score`.  
- *Expectation:* $R^2$ or F1—trees often outperform linear models on raw data (e.g., 0.75+).

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score

# Same simulated dataset
np.random.seed(42)
n_samples = 1000
data = {
    'num1': np.random.normal(0, 1, n_samples),
    'num2': np.random.normal(10, 5, n_samples),
    'num3': np.random.randint(1, 4, n_samples),  # Low variance
    'num4': np.ones(n_samples) + np.random.normal(0, 0.01, n_samples),  # Near-constant
    'num5': np.random.uniform(0, 100, n_samples),
    'num6': np.random.randint(1, 4, n_samples),  # Low variance
    'num7': np.random.normal(50, 2, n_samples),
    'cat1': np.random.choice(['yes', 'no'], n_samples),  # Low cardinality
    'cat2': np.random.choice(['red', 'blue', 'green'], n_samples),  # Medium
    'cat3': np.random.choice([f'state_{i}' for i in range(10)], n_samples)  # High cardinality
}
X = pd.DataFrame(data)
y = X['num1'] + 2 * X['num3'] + np.where(X['cat1'] == 'yes', 1, 0) + np.random.normal(0, 1, n_samples)

# Step 1: Handle Categorical Variables
# Label encode all categoricals (trees handle high cardinality)
cat_cols = ['cat1', 'cat2', 'cat3']
X_encoded = X.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    label_encoders[col] = le  # Save for later use
print(f"Shape after encoding: {X_encoded.shape}")  # 1000 x 10 (no column explosion)

# Step 2: Variance Threshold
selector = VarianceThreshold(threshold=0.01)  # Low threshold for 1,2,3-like variables
X_filtered = selector.fit_transform(X_encoded)
kept_features = X_encoded.columns[selector.get_support()].tolist()
print(f"Shape after variance filter: {X_filtered.shape}")  # e.g., 1000 x 9 (drops num4)
print(f"Kept features: {len(kept_features)}")

# Convert back to DataFrame
X_filtered_df = pd.DataFrame(X_filtered, columns=kept_features)

# Step 3: No Standardization (trees don’t need it)
X_ready = X_filtered_df.copy()
print(f"Shape (no scaling): {X_ready.shape}")

# Step 4: Train Tree Model
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)  # Limit depth to avoid overfitting
rf.fit(X_ready, y)
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({'feature': kept_features, 'importance': importances})
print(feature_importance_df.sort_values('importance', ascending=False))

# Step 5: Feature Selection
# Select top features (e.g., top 5 or 90% cumulative importance)
threshold = 0.05  # Arbitrary importance threshold
selected_features = feature_importance_df[feature_importance_df['importance'] > threshold]['feature']
X_selected = X_ready[selected_features]
print(f"Shape after feature selection: {X_selected.shape}")  # e.g., 1000 x 4
print(f"Selected features: {selected_features.tolist()}")

# Step 6: Model & Evaluate
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.3, random_state=42)
rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_final.fit(X_train, y_train)
cv_scores = cross_val_score(rf_final, X_train, y_train, cv=5, scoring='r2')
test_score = rf_final.score(X_test, y_test)
print(f"Cross-validated R²: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
print(f"Test R²: {test_score:.3f}")

Shape after encoding: (1000, 10)
Shape after variance filter: (1000, 9)
Kept features: 9
Shape (no scaling): (1000, 9)
  feature  importance
2    num3    0.534777
0    num1    0.277121
6    cat1    0.046970
1    num2    0.036137
5    num7    0.035588
3    num5    0.033050
8    cat3    0.019798
7    cat2    0.008675
4    num6    0.007884
Shape after feature selection: (1000, 2)
Selected features: ['num1', 'num3']
Cross-validated R²: 0.694 (±0.042)
Test R²: 0.610


**Key Differences**

*PCA + Lasso:* Great for shrinking to synthetic features, handles numerical/categorical mix, but loses interpretability. High cardinality bloats encoding—manage with variance cuts.

*Trees:* Keeps original features, simpler preprocessing, robust to cardinality, but may overfit high-cardinality splits—tune depth/pruning.

| **Step**            | **PCA + Lasso**               | **Tree-Based**              |
|---------------------|-------------------------------|-----------------------------|
| **Categorical**     | One-hot (explodes columns)    | Label encode (keeps compact) |
| **Low Variance**    | Keep with low threshold       | Keep—trees split anyway     |
| **High Cardinality**| One-hot or target encode      | Label encode, watch bias    |
| **Scaling**         | Required (standardize)        | Not needed                  |
| **Reduction**       | PCA (synthetic) $\to$ Lasso   | Tree importance (original)  |
| **Output**          | $n \times 30$ PCs             | $n \times 30$ features      |